In [ ]:
"""
OPTUNA MASTERY SCRIPT
======================
This script builds on your original Random Forest + Iris example and layers in
every important Optuna concept, one piece at a time. Read the comments — they
explain the WHY, not just the WHAT.
"""

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import optuna


# ---------------------------------------------------------------------------
# STEP 1: Proper data splitting (fixes the "data leakage" issue)
# ---------------------------------------------------------------------------
X, y = load_iris(return_X_y=True)

# First split: separate out a final, untouched test set (20% of all data)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: from the remaining 80%, carve out a validation set (25% of
# that 80% = 20% of the original total), leaving 60% for actual training.
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

# End result roughly: 60% train / 20% validation / 20% test


# ---------------------------------------------------------------------------
# STEP 2: The objective function
# ---------------------------------------------------------------------------
def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])
    min_samples_split = trial.suggest_float("min_samples_split", 0.01, 0.5)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        criterion=criterion,
        min_samples_split=min_samples_split,
        random_state=42,
    )

    scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="accuracy")
    mean_accuracy = scores.mean()

    return mean_accuracy


# ---------------------------------------------------------------------------
# STEP 3: Run the study
# ---------------------------------------------------------------------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best hyperparameters found:", study.best_params)
print("Best cross-validated accuracy during tuning:", study.best_value)


# ---------------------------------------------------------------------------
# STEP 4: The ONE-TIME final evaluation on the untouched test set
# ---------------------------------------------------------------------------
best_params = study.best_params

final_clf = RandomForestClassifier(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    criterion=best_params["criterion"],
    min_samples_split=best_params["min_samples_split"],
    random_state=42,
)

import numpy as np
X_final_train = np.concatenate([X_train, X_val])
y_final_train = np.concatenate([y_train, y_val])

final_clf.fit(X_final_train, y_final_train)
final_pred = final_clf.predict(X_test)
final_accuracy = accuracy_score(y_test, final_pred)

print("Final, honest test-set accuracy:", final_accuracy)


# ---------------------------------------------------------------------------
# STEP 5: Inspecting what happened
# ---------------------------------------------------------------------------
df = study.trials_dataframe()
print(df.head())

# optuna.visualization.plot_optimization_history(study)
# optuna.visualization.plot_param_importances(study)
# optuna.visualization.plot_parallel_coordinate(study)